In [1]:
import pandas as pd

In [2]:
dss = ["qnli", "mrpc", "qqp", "sst2", "cola", "mnli", "rte", "stsb"]

In [5]:
%cd ..

/home/dvitel/data/DataInf


In [10]:
dfs = {}
for ds in dss:
    dfs[ds] = pd.read_csv(f"./data/llama/infl-time/{ds}.csv", header=None, names=["task", "method", "seed", "prep_time", "grad_time", "calc_time"])
    pass

one_ds = pd.concat(dfs.values(), ignore_index=True)

In [23]:
one_ds_m = one_ds.melt(id_vars=["task", "method", "seed"], value_vars=["prep_time", "grad_time", "calc_time"], value_name="time", var_name="time_type")

In [19]:
one_ds_m

,task,method,seed,time_type,time
0,qnli,datainf,0,prep_time,0.00000
1,qnli,datainf,1,prep_time,0.00000
2,qnli,datainf,2,prep_time,0.00000
3,qnli,datainf,3,prep_time,0.00000
4,qnli,datainf,4,prep_time,0.00000
...,...,...,...,...,...
1435,stsb,hf_we_,5,calc_time,335.17291
1436,stsb,hf_we_,6,calc_time,336.22370
1437,stsb,hf_we_,7,calc_time,333.80965
1438,stsb,hf_we_,8,calc_time,335.60467


In [25]:
print(one_ds_m.columns.tolist())

['task', 'method', 'seed', 'time_type', 'time']


In [27]:
one_ds_p = one_ds_m.pivot_table(index=["method", "seed"], columns=["task", "time_type"], values="time", aggfunc="mean")

In [33]:
one_ds_p = one_ds_p.sort_index(level=["method", "seed"]).reindex(dss, level="task", axis=1).reindex(["prep_time", "grad_time", "calc_time"], level=1, axis=1)


In [36]:
final_df = one_ds_p.groupby("method").mean()
final_df

task                qnli                              mrpc              \
time_type      prep_time   grad_time   calc_time prep_time   grad_time   
method                                                                   
cos             0.000000  475.701705    0.265342  0.000000  294.188225   
datainf         0.000000  131.484505    0.151001  0.000000  107.442337   
hf              0.000000  473.045332    0.057949  0.000000  309.289793   
hf_we_         10.535227  670.332046  467.238454  1.923975  286.738237   
hf_we_topk_10  17.055116  670.616168  406.518504  3.881848  283.623237   
outlier         0.000000  130.287418  215.211718  0.000000  105.831488   

task                             qqp                              sst2  ...  \
time_type       calc_time  prep_time   grad_time   calc_time prep_time  ...   
method                                                                  ...   
cos              0.235043   0.000000  355.751380    0.414841  0.000000  ...   
datainf          0.081620   0.000000  128.788242    0.033271  0.000000  ...   
hf               0.007247   0.000000  365.461745    0.007630  0.000000  ...   
hf_we_         115.320351   6.673066  448.027273  311.732051  5.100069  ...   
hf_we_topk_10  140.545979  10.471464  446.983770  407.810918  7.723798  ...   
outlier         82.109554   0.000000  128.996267  218.739138  0.000000  ...   

task                 cola       mnli                               rte  \
time_type       calc_time  prep_time   grad_time   calc_time prep_time   
method                                                                   
cos              0.274753   0.000000  468.397029    0.295509  0.000000   
datainf          0.100967   0.000000  128.278932    0.064174  0.000000   
hf               0.161213   0.000000  466.847643    0.043380  0.000000   
hf_we_         193.925696   8.966883  581.389887  384.334574  2.040258   
hf_we_topk_10  418.747203  13.141194  565.536066  406.250868  4.603571   
outlier        251.595225   0.000000  128.773710  218.227428  0.000000   

task                                        stsb                          
time_type       grad_time   calc_time  prep_time   grad_time   calc_time  
method                                                                    
cos            202.042081    0.391289   0.000000  361.652368    0.486658  
datainf         72.179140    0.037919   0.000000  131.764548    0.108210  
hf             200.657688    0.004220   0.000000  360.611705    0.007943  
hf_we_         191.126034  119.949069   6.978768  448.114867  335.118776  
hf_we_topk_10  189.028381  124.816226  10.727678  449.259486  400.847917  
outlier         72.802204   67.701853   0.000000  129.032083  216.995386  

[6 rows x 24 columns]

In [35]:
from tabulate import tabulate

In [39]:
final_df.round(2).reset_index()

task              method      qnli                          mrpc            \
time_type                prep_time grad_time calc_time prep_time grad_time   
0                    cos      0.00    475.70      0.27      0.00    294.19   
1                datainf      0.00    131.48      0.15      0.00    107.44   
2                     hf      0.00    473.05      0.06      0.00    309.29   
3                 hf_we_     10.54    670.33    467.24      1.92    286.74   
4          hf_we_topk_10     17.06    670.62    406.52      3.88    283.62   
5                outlier      0.00    130.29    215.21      0.00    105.83   

task                      qqp                      ...      cola      mnli  \
time_type calc_time prep_time grad_time calc_time  ... calc_time prep_time   
0              0.24      0.00    355.75      0.41  ...      0.27      0.00   
1              0.08      0.00    128.79      0.03  ...      0.10      0.00   
2              0.01      0.00    365.46      0.01  ...      0.16      0.00   
3            115.32      6.67    448.03    311.73  ...    193.93      8.97   
4            140.55     10.47    446.98    407.81  ...    418.75     13.14   
5             82.11      0.00    129.00    218.74  ...    251.60      0.00   

task                                rte                          stsb  \
time_type grad_time calc_time prep_time grad_time calc_time prep_time   
0            468.40      0.30      0.00    202.04      0.39      0.00   
1            128.28      0.06      0.00     72.18      0.04      0.00   
2            466.85      0.04      0.00    200.66      0.00      0.00   
3            581.39    384.33      2.04    191.13    119.95      6.98   
4            565.54    406.25      4.60    189.03    124.82     10.73   
5            128.77    218.23      0.00     72.80     67.70      0.00   

task                           
time_type grad_time calc_time  
0            361.65      0.49  
1            131.76      0.11  
2            360.61      0.01  
3            448.11    335.12  
4            449.26    400.85  
5            129.03    217.00  

[6 rows x 25 columns]

In [38]:
markdown = tabulate(final_df.round(2).reset_index(), headers='keys', tablefmt='github')
print(markdown)

|    | ('method', '')   |   ('qnli', 'prep_time') |   ('qnli', 'grad_time') |   ('qnli', 'calc_time') |   ('mrpc', 'prep_time') |   ('mrpc', 'grad_time') |   ('mrpc', 'calc_time') |   ('qqp', 'prep_time') |   ('qqp', 'grad_time') |   ('qqp', 'calc_time') |   ('sst2', 'prep_time') |   ('sst2', 'grad_time') |   ('sst2', 'calc_time') |   ('cola', 'prep_time') |   ('cola', 'grad_time') |   ('cola', 'calc_time') |   ('mnli', 'prep_time') |   ('mnli', 'grad_time') |   ('mnli', 'calc_time') |   ('rte', 'prep_time') |   ('rte', 'grad_time') |   ('rte', 'calc_time') |   ('stsb', 'prep_time') |   ('stsb', 'grad_time') |   ('stsb', 'calc_time') |
|----|------------------|-------------------------|-------------------------|-------------------------|-------------------------|-------------------------|-------------------------|------------------------|------------------------|------------------------|-------------------------|-------------------------|-------------------------|----------------------